# UKCP18: quickstart

Download monthly mean temperature from the UKCP18 regional 12 km
projections (ensemble member 01, RCP8.5) and plot a map of projected
temperature change over the UK.

**Prerequisites:**
- Free CEDA account: https://services.ceda.ac.uk/cedasite/register/info/
- CEDA bearer token set as `CEDA_TOKEN` env var or saved to `~/.ceda_token`
- `pip install requests xarray netcdf4 matplotlib cartopy numpy`

See [`docs/ukcp18/README.md`](../docs/ukcp18/README.md) for the full
reference.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLE = "tas"          # Mean air temperature at 1.5 m
SCENARIO = "rcp85"
COLLECTION = "land-rcm"
DOMAIN = "uk"
RESOLUTION = "12km"
ENSEMBLE_MEMBER = "01"
FREQUENCY = "mon"
VERSION = "v20190731"
TIME_PERIOD = "19801201-20801130"

# Baseline and future periods for comparison
BASELINE = slice("1981", "2000")
FUTURE = slice("2061", "2080")

OUTPUT_DIR = "../data/ukcp18"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["requests", "xarray", "netcdf4", "matplotlib", "cartopy"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Download monthly mean temperature

Downloads from the CEDA archive using a bearer token. The regional
12 km monthly file covers 1980-2080 in a single NetCDF.

In [ ]:
from scripts.ukcp18_download import download  # noqa: E402

nc_path = download(
    variable=VARIABLE,
    scenario=SCENARIO,
    collection=COLLECTION,
    domain=DOMAIN,
    resolution=RESOLUTION,
    ensemble_member=ENSEMBLE_MEMBER,
    frequency=FREQUENCY,
    version=VERSION,
    time_period=TIME_PERIOD,
    output_dir=OUTPUT_DIR,
)
print(f"NetCDF: {nc_path} ({nc_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

The UK-regridded files use projection_y_coordinate and
projection_x_coordinate on the OSGB grid (EPSG:27700).

In [ ]:
ds = xr.open_dataset(nc_path)
print(ds)

print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Grid shape: {ds[VARIABLE].shape}")

## Plot projected temperature change

Compute the difference between a future period (2061-2080) and a
baseline period (1981-2000) to show the projected warming across
the UK under RCP8.5.

In [ ]:
baseline_mean = ds[VARIABLE].sel(time=BASELINE).mean("time")
future_mean = ds[VARIABLE].sel(time=FUTURE).mean("time")
warming = future_mean - baseline_mean

# The OSGB-regridded files include latitude and longitude as 2D fields
if "latitude" in ds and "longitude" in ds:
    lons = ds["longitude"].values
    lats = ds["latitude"].values
    fig = plt.figure(figsize=(8, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    im = ax.pcolormesh(
        lons, lats, warming.values,
        transform=ccrs.PlateCarree(),
        cmap="RdYlBu_r",
        vmin=0, vmax=5,
    )
else:
    # Fall back to OSGB axes if no lat/lon fields
    osgb = ccrs.OSGB()
    fig = plt.figure(figsize=(8, 10))
    ax = plt.axes(projection=osgb)
    im = ax.pcolormesh(
        warming.coords[list(warming.dims)[-1]].values,
        warming.coords[list(warming.dims)[-2]].values,
        warming.values,
        transform=osgb,
        cmap="RdYlBu_r",
        vmin=0, vmax=5,
    )

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")

cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label("Temperature change (degC)")

ax.set_title(
    f"UKCP18 regional 12km, member {ENSEMBLE_MEMBER}\n"
    f"Projected warming: {FUTURE.start}-{FUTURE.stop} vs {BASELINE.start}-{BASELINE.stop} (RCP8.5)"
)
plt.tight_layout()
plt.show()

print(f"Mean warming across UK: {float(warming.mean()):.2f} degC")
print(f"Max warming: {float(warming.max()):.2f} degC")

## Next steps

- **Full reference**: [`docs/ukcp18/README.md`](../docs/ukcp18/README.md)
- **Daily data**: change `FREQUENCY = "day"` for daily fields (larger
  download, different time period strings).
- **Other variables**: try `pr` (precipitation), `tasmax` (daily
  maximum temperature), `sfcWind` (wind speed).
- **Ensemble spread**: download members 01 through 12 and compute the
  ensemble mean and range to quantify projection uncertainty.
- **Probabilistic strand**: use `land-prob` for multi-scenario analysis
  (RCP2.6, 4.5, 6.0, 8.5) at 25 km resolution.
- **Bias correction**: see
  [Sheridan et al. 2025](https://essd.copernicus.org/articles/17/2113/2025/)
  for bias-corrected UKCP18 data.
- **UKCP18 user interface**: explore interactively at
  https://ukclimateprojections-ui.metoffice.gov.uk/